In [ ]:
# RAILPULSE | NB_04_Live_Simulator  (une seule cellule, aucun lakehouse nécessaire)
# Collez ici la chaîne de connexion du point de terminaison personnalisé
CONNECTION_STRING = "COLLEZ_ICI_LA_CHAINE_DE_CONNEXION_CLE_PRIMAIRE"
DURATION_MIN, TICK_SEC = 30, 3

import subprocess, sys   # installation sans redémarrage du noyau
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "azure-eventhub"])

import json, time, math, random, datetime as dt
from zoneinfo import ZoneInfo
from azure.eventhub import EventHubProducerClient, EventData
fr = lambda n: f"{n:,}".replace(",", " ")   # 98 950 plutôt que 98,950

S = {"PAR": (48.8768, 2.3592), "LUX": (49.6000, 6.1339), "MTZ": (49.1097, 6.1772), "STR": (48.5850, 7.7350),
     "BRU": (50.8357, 4.3360), "AMS": (52.3791, 4.9003), "CGN": (50.9430, 6.9589), "FRA": (50.1071, 8.6638),
     "BSL": (47.5476, 7.5897), "ZRH": (47.3779, 8.5403), "LYS": (45.7606, 4.8594), "MIL": (45.4864, 9.2048)}
L = [("L01", "PAR", "LUX", 125, 6, 510), ("L02", "PAR", "STR", 106, 10, 510), ("L03", "PAR", "BRU", 82, 12, 450),
     ("L04", "BRU", "AMS", 110, 10, 450), ("L05", "BRU", "CGN", 115, 7, 450), ("L06", "CGN", "FRA", 64, 14, 460),
     ("L07", "LUX", "MTZ", 45, 20, 300), ("L08", "STR", "FRA", 110, 6, 460), ("L09", "STR", "BSL", 80, 12, 300),
     ("L10", "BSL", "ZRH", 55, 16, 400), ("L11", "PAR", "LYS", 117, 14, 510), ("L12", "ZRH", "MIL", 197, 8, 420)]

tz = ZoneInfo("Europe/Paris")
real_start = dt.datetime.now(tz)
# Hors service (nuit) : on rejoue la pointe de 17 h 30 pour toujours avoir des trains sur le réseau
sim_start = real_start if 7 <= real_start.hour < 21 else real_start.replace(hour=17, minute=30, second=0)
random.seed(42)
drift = {}

def running(now):
    trains = []
    for code, a, b, mins, n, seats in L:
        for o, d in ((a, b), (b, a)):
            for k in range(n):
                h = 6 + k * 15 / max(n - 1, 1)
                dep = now.replace(hour=int(h), minute=int(round((h % 1) * 60 / 5) * 5) % 60, second=0, microsecond=0)
                p = (now - dep).total_seconds() / 60 / mins
                if 0 <= p <= 1:
                    trains.append((f"{now:%Y%m%d}-{o}{d}-{dep:%H%M}", code, o, d, p, seats))
    return trains

producer = EventHubProducerClient.from_connection_string(CONNECTION_STRING)
sent, ticks = 0, int(DURATION_MIN * 60 / TICK_SEC)
for i in range(ticks):
    now = sim_start + (dt.datetime.now(tz) - real_start)
    batch = producer.create_batch()
    for tid, code, o, d, p, seats in running(now):
        drift[tid] = min(12, max(0, drift.get(tid, random.choice([0, 0, 1, 2, 4])) + random.choice([-1] + [0] * 8 + [1])))
        incident = code == "L02" and i > ticks // 5          # incident technique simulé sur Paris > Strasbourg
        delay = drift[tid] + (min(40, (i - ticks // 5) // 4) if incident else 0)
        (la1, lo1), (la2, lo2) = S[o], S[d]
        speed = 0 if incident and delay > 10 else round(random.gauss(285 if seats >= 450 else 150, 12))
        evt = {"event_time": dt.datetime.now(dt.timezone.utc).isoformat(), "train_id": tid, "line_code": code,
               "origin": o, "destination": d, "progress_pct": round(100 * p, 1),
               "latitude": round(la1 + (la2 - la1) * p, 5), "longitude": round(lo1 + (lo2 - lo1) * p, 5),
               "speed_kmh": speed, "delay_min": delay,
               "passengers": int(seats * random.uniform(0.45, 0.95)),
               "energy_kwh": round(max(speed, 0) * random.uniform(0.18, 0.24), 1),
               "status": "INCIDENT" if incident and delay > 10 else "EN_ROUTE"}
        batch.add(EventData(json.dumps(evt)))
        sent += 1
    producer.send_batch(batch)
    if i % 20 == 0:
        print(f"{now:%H:%M:%S} | trains en circulation : {len(running(now))} | événements envoyés : {fr(sent)}")
    time.sleep(TICK_SEC)
producer.close()
print(f"Terminé : {fr(sent)} événements envoyés.")